In [0]:

from datetime import datetime

modo = "historico" # "automatico"


if modo == "automatico":
  periodo = datetime.now().strftime("%m-%Y")
else :
  periodo = None

In [0]:
tb_full = [
    "TB_CLIENTES_CORE",
    "TB_PRODUCTOS_CAT",
    "TB_SUCURSALES_RED"
]

tb_incremental = {
    "TB_COMISIONES_LOG" : "fec_cobro",
    "TB_MOV_FINANCIEROS" : "fec_mov",
    "TB_OBLIGACIONES" : "fec_desembolso"
}


In [0]:
import hashlib
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

PATH_BRONZE = "abfss://bronze@stdataknowdeveastus001.dfs.core.windows.net/"
PATH_SILVER = "abfss://silver@stdataknowdeveastus001.dfs.core.windows.net/"

tb_full = [
    "TB_CLIENTES_CORE",
    "TB_PRODUCTOS_CAT",
    "TB_SUCURSALES_RED"
]

# Columnas PII a enmascarar (aplica a cualquier tabla que las tenga)
PII_COLS = {"nomb_cli", "apell_cli", "num_doc"}

# UDF SHA-256
hash_udf = F.udf(
    lambda v: hashlib.sha256(str(v).encode()).hexdigest() if v is not None else None,
    StringType()
)


def transformar(df, tabla):
    total = df.count()

    # 1. Duplicados exactos
    df = df.dropDuplicates()

    # 2. Registros con cualquier campo nulo -> tabla de errores
    condicion_nulo = F.lit(False)
    for col in df.columns:
        condicion_nulo = condicion_nulo | F.col(col).isNull()

    df_errores = (
        df.filter(condicion_nulo)
          .withColumn("motivo", F.lit("campo nulo en columna obligatoria"))
          .withColumn("tabla_origen", F.lit(tabla))
          .withColumn("fecha_rechazo", F.current_timestamp())
    )
    df = df.filter(~condicion_nulo)

    # 3. Estandarizar tipos
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.upper(F.col(col_name))))
        if "fecha" in col_name.lower() or "date" in col_name.lower():
            df = df.withColumn(col_name, F.to_date(F.col(col_name)))

    # 4. Enmascarar PII
    for col_name in df.columns:
        if col_name.lower() in PII_COLS:
            df = df.withColumn(col_name, hash_udf(F.col(col_name)))

    # 5. Reporte de calidad
    conformes = df.count()
    rechazados = total - conformes
    print(f"\n{'='*50}")
    print(f"CALIDAD: {tabla}")
    print(f"  Originales : {total}")
    print(f"  Rechazados : {rechazados}")
    print(f"  Conformes  : {conformes} ({conformes/total*100:.1f}%)")
    print(f"  Nulos por columna:")
    for col_name in df.columns:
        n = df.filter(F.col(col_name).isNull()).count()
        print(f"    {col_name:<30} {n/conformes*100:.1f}%")
    print(f"{'='*50}\n")

    return df, df_errores


def cargue_full(list_full):
    for tabla in list_full:
        df = spark.read.format("parquet").load(f"{PATH_BRONZE}/{tabla}")

        df_clean, df_errores = transformar(df, tabla)

        # Guardar errores
        if df_errores.count() > 0:
            (df_errores.write
             .mode("append")
             .format("delta")
             .save(f"{PATH_SILVER}/errors/{tabla}"))

        # Guardar Silver
        (df_clean.write
         .mode("overwrite")
         .format("delta")
         .option("overwriteSchema", "true")
         .save(f"{PATH_SILVER}/cleaned/{tabla}"))

        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS silver.cleaned.{tabla}
            USING DELTA LOCATION '{PATH_SILVER}/cleaned/{tabla}'
        """)


cargue_full(tb_full)

In [0]:
import hashlib
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

PATH_BRONZE = "abfss://bronze@stdataknowdeveastus001.dfs.core.windows.net/"
PATH_SILVER = "abfss://silver@stdataknowdeveastus001.dfs.core.windows.net/"

tb_incremental = {
    "TB_COMISIONES_LOG"  : "fec_cobro",
    "TB_MOV_FINANCIEROS" : "fec_mov",
    "TB_OBLIGACIONES"    : "fec_desembolso"
}

PII_COLS = {"nomb_cli", "apell_cli", "num_doc"}

hash_udf = F.udf(
    lambda v: hashlib.sha256(str(v).encode()).hexdigest() if v is not None else None,
    StringType()
)

# ─────────────────────────────────────────────
# MODO DE EJECUCION
# ─────────────────────────────────────────────

modo = "historico"  # "historico"

if modo == "automatico":
    periodo = datetime.now().strftime("%m-%Y")
else:
    periodo = None


# ─────────────────────────────────────────────
# UTILIDADES
# ─────────────────────────────────────────────

def periodos_en_bronze(tabla):
    try:
        carpetas = dbutils.fs.ls(f"{PATH_BRONZE}/{tabla}/")
        return [f.name.strip("/") for f in carpetas]
    except Exception:
        return []


def ruta_existe(ruta):
    try:
        archivos = dbutils.fs.ls(ruta)
        return any(f.name.endswith(".parquet") or f.name.endswith(".snappy.parquet") for f in archivos)
    except Exception:
        return False


# ─────────────────────────────────────────────
# TRANSFORMACIONES
# ─────────────────────────────────────────────

def transformar(df, tabla, periodo):
    total = df.count()

    # 1. Duplicados
    df = df.dropDuplicates()

    # 2. Nulos -> errores
    condicion_nulo = F.lit(False)
    for col in df.columns:
        condicion_nulo = condicion_nulo | F.col(col).isNull()

    df_errores = (
        df.filter(condicion_nulo)
          .withColumn("motivo", F.lit("campo nulo en columna obligatoria"))
          .withColumn("tabla_origen", F.lit(tabla))
          .withColumn("fecha_rechazo", F.current_timestamp())
    )
    df = df.filter(~condicion_nulo)

    # 3. Estandarizar tipos
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.upper(F.col(col_name))))
        if "fecha" in col_name.lower() or "date" in col_name.lower():
            df = df.withColumn(col_name, F.to_date(F.col(col_name), "dd-MM-yyyy"))

    # 4. Enmascarar PII
    for col_name in df.columns:
        if col_name.lower() in PII_COLS:
            df = df.withColumn(col_name, hash_udf(F.col(col_name)))

    # 5. Columna de particion
    df = df.withColumn("periodo", F.lit(periodo))

    # 6. Reporte de calidad
    conformes = df.count()
    rechazados = total - conformes
    pct = conformes / total * 100 if total > 0 else 0
    print(f"\n{'='*50}")
    print(f"CALIDAD: {tabla} | Periodo: {periodo}")
    print(f"  Originales : {total}")
    print(f"  Rechazados : {rechazados}")
    print(f"  Conformes  : {conformes} ({pct:.1f}%)")
    print(f"  Nulos por columna:")
    for col_name in df.columns:
        n = df.filter(F.col(col_name).isNull()).count()
        print(f"    {col_name:<30} {n / conformes * 100:.1f}%" if conformes > 0 else f"    {col_name:<30} N/A")
    print(f"{'='*50}\n")

    return df, df_errores


# ─────────────────────────────────────────────
# PIPELINE
# ─────────────────────────────────────────────

def procesar_periodo(df, tabla, periodo):
    df_clean, df_errores = transformar(df, tabla, periodo)

    if df_errores.count() > 0:
        (df_errores.write
         .mode("append")
         .format("delta")
         .save(f"{PATH_SILVER}/errors/{tabla}"))

    ruta_silver = f"{PATH_SILVER}/cleaned/{tabla}"

    if DeltaTable.isDeltaTable(spark, ruta_silver):
        spark.sql(f"DELETE FROM delta.`{ruta_silver}` WHERE periodo = '{periodo}'")
        df_clean.write.mode("append").format("delta").save(ruta_silver)
    else:
        (df_clean.write
         .mode("overwrite")
         .format("delta")
         .partitionBy("periodo")
         .save(ruta_silver))
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS silver.cleaned.{tabla}
            USING DELTA LOCATION '{ruta_silver}'
        """)


def cargue_incremental(diccionario):
    print(f"\nModo: {modo.upper()} | Periodo: {periodo or 'todos'}\n")

    for tabla, col_fecha in diccionario.items():
        periodos = [periodo] if periodo else periodos_en_bronze(tabla)

        for p in periodos:
            ruta = f"{PATH_BRONZE}/{tabla}/{p}"

            if not ruta_existe(ruta):
                print(f"[SKIP] {tabla}/{p}: sin archivos.")
                continue

            df = spark.read.option("mergeSchema", "true").format("parquet").load(ruta)
            procesar_periodo(df, tabla, p)


cargue_incremental(tb_incremental)